# Transformer LM

transformer_lm.py

**element-wise multiplies:**

**dot product:**

`Num of matrix multiplies: num_layers * 9 matrix multiplies + 1`

`FLOPs: (num_layers * (((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * seq_len * d_model * seq_len) + (batch...) * (2 * seq_len * seq_len * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * seq_len * d_model * d_ff) + (batch...) * (2 * seq_len * d_model * d_ff) + (batch...) * (2 * seq_len * d_ff * d_model)))) + (batch...) * (2 * seq_len * d_model * vocab_size) FLOPs`

* init: `0 matrix multiplies` `0 FLOPs`
* forward:
    * Num of matrix multiplies: `num_layers * 9 matrix multiplies + 1`
    * FLOPs: `(num_layers * (((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * seq_len * d_model * seq_len) + (batch...) * (2 * seq_len * seq_len * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * seq_len * d_model * d_ff) + (batch...) * (2 * seq_len * d_model * d_ff) + (batch...) * (2 * seq_len * d_ff * d_model)))) + (batch...) * (2 * seq_len * d_model * vocab_size) FLOPs`

**trainable parameters:** `vocab_size * d_model + num_layers * (d_model + d_model + 4 * (d_model * d_model) + 3 * (d_ff * d_model)) + d_model + vocab_size * d_model`
* init:
  * self.embedding_layer: Parameters: vocab_size * d_model
  * self.transformerblock_layer: Parameters: num_layers * (d_model + d_model + 4 * (d_model * d_model) + 3 * (d_ff * d_model))
  * self.rmsnorm_layer: Parameters: d_model
  * self.linear_layer: Parameters: vocab_size * d_model

* forward: 0 parameters

In [ ]:
import torch.nn as nn
from jaxtyping import Float, Int
from torch import Tensor
from cs336_basics.embedding import Embedding
from cs336_basics.transformer_block import TransformerBlock
from cs336_basics.rmsnorm_einx import RMSNorm
from cs336_basics.linear_module import Linear

class TransformerLM(nn.Module):
    def __init__(self, vocab_size: int, context_length: int, num_layers: int, d_model: int, num_heads: int, d_ff: int, rope_theta: float):
        """Implement the Transformer language model

            Args:
                vocab_size (int): The size of the vocabulary, necessary for determining the dimensionality of the token embedding matrix
                context_length (int): The maximum context length, necessary for determining the dimensionality of the position embedding matrix
                num_layers (int): The number of Transformer blocks to use
                d_model (int): Dimensionality of the Transformer block inputs
                num_heads (int): Number of heads to use in multi-head self-attention
                d_ff (int): Dimensionality of the position-wise feed-forward inner layer
                rope_theta (float): RoPE parameter
        """
        super().__init__()

        self.embedding_layer = Embedding(vocab_size, d_model) # 0 matrix multiplies; 0 FLOPs | Parameters: vocab_size * d_model
        self.transformerblock_layer = nn.ModuleList(
            [TransformerBlock(d_model, num_heads, d_ff, context_length, rope_theta) for _ in range(num_layers)]
        ) # num_layers * (0 matrix multiplies; 0 FLOPs) | Parameters: num_layers * (d_model + d_model + 4 * (d_model * d_model) + 3 * (d_ff * d_model))
        # self.transformerblock_layer = [TransformerBlock(d_model, num_heads, d_ff, context_length, rope_theta) for _ in range(num_layers)]
        self.rmsnorm_layer = RMSNorm(d_model) # 0 matrix multiplies; 0 FLOPs | Parameters: d_model
        self.linear_layer = Linear(d_model, vocab_size) # 0 matrix multiplies; 0 FLOPs | Parameters: vocab_size * d_model
    def forward(self, in_indices: Int[Tensor, " batch_size sequence_length"]) -> Float[Tensor, " batch_size sequence_length vocab_size"]:
        """Implement the Transformer language model

            Args:
                in_indices (Int[Tensor, " batch_size sequence_length"]): Tensor with input indices to run the language model on. Shape is (batch_size, sequence_length), where
            `sequence_length` is at most `context_length`

            Returns:
                Float[Tensor, "batch_size sequence_length vocab_size"]: Tensor with the predicted unnormalized next-word distribution for each token.
        """
        input_embedding = self.embedding_layer.forward(in_indices) # 0 matrix multiplies; 0 FLOPs | Parameters: 0 parameters

        for block in self.transformerblock_layer:                  # num_layers * (9 matrix multiplies; ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * seq_len * d_model * seq_len) + (batch...) * (2 * seq_len * seq_len * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_ff * d_model) FLOPs) | Parameters: num_layers * 0 parameters
            output_embedding = block.forward(input_embedding)
            input_embedding = output_embedding

        output_embedding = self.rmsnorm_layer.forward(output_embedding) # 0 matrix multiplies; 0 FLOPs | Parameters: 0 parameters

        result = self.linear_layer.forward(output_embedding) # 1 matrix multiplies; FLOPs = (batch...) * (2 * seq_len * d_model * vocab_size) FLOPs | Parameters: 0 parameters

        return result

## TokenEmbedding

embedding.py

**dot product:**

* Num of matrix multiplies: `0 matrix multiplies`

* FLOPs: `0 FLOPs`

**trainable parameters:** vocab_size * d_model
* init:
  * self.W(vocab_size, d_model)
* forward: 0 parameters

In [ ]:
import torch
import torch.nn as nn

class Embedding(nn.Module):
    def __init__(self, num_embeddings: int, embedding_dim: int, device: torch.device | None = None, dtype: torch.dtype | None = None):
        """Construct an embedding module.

        Args:
            num_embeddings(int): Size of the vocabulary
            embedding_dim(int): Dimension of the embedding vectors
            device(torch.device | None): Device to store the parameters on
            dtype(torch.dtype | None): Data type of the parameters
        """
        super().__init__()
        temp_W = torch.empty(num_embeddings, embedding_dim, device=device, dtype=dtype)
        temp_W = torch.nn.init.trunc_normal_(temp_W, mean=0, std=1, a=-3, b=3)
        self.W = nn.Parameter(temp_W) # Parameter: self.W(vocab_size, d_model)
    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        """Lookup the embedding vectors for the given token IDs.

        Args:
            token_ids(torch.Tensor): token ids with shape (batch_size, sequence_length)
        """
        return self.W[token_ids]

## Transformer Block
transformer_block.py

**dot product:**

`Num of matrix multiplies: 9 matrix multiplies`;

`FLOPs: ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * seq_len * d_model * seq_len) + (batch...) * (2 * seq_len * seq_len * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * seq_len * d_model * d_ff) + (batch...) * (2 * seq_len * d_model * d_ff) + (batch...) * (2 * seq_len * d_ff * d_model) FLOPs`

* init: `0 matrix multiplies`; `0 FLOPs`
* forward: `9 matrix multiplies`;`((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * seq_len * d_model * seq_len) + (batch...) * (2 * seq_len * seq_len * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * seq_len * d_model * d_ff) + (batch...) * (2 * seq_len * d_model * d_ff) + (batch...) * (2 * seq_len * d_ff * d_model) FLOPs`

**trainable parameters:** `d_model + d_model + 4 * (d_model * d_model) + 3 * (d_ff * d_model)`
* init:
  * self.rmsnorm_first_layer(Parameter: (d_model))
  * self.rmsnorm_second_layer(Parameter: (d_model))
  * self.multiheadselfattentionrops_layer(Parameter: 4 * (d_model * d_model))
  * self.positionwiseffn_layer(Parameter: 3 * (d_ff * d_model))
* forward: 0 parameters






In [ ]:
import torch
import torch.nn as nn
from jaxtyping import Float
from torch import Tensor
from cs336_basics.rmsnorm_einx import RMSNorm
from cs336_basics.multihead_self_attention_rope import MultiHeadSelfAttentionRope
from cs336_basics.positionwise_feedforward_einx import PWFFN

class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, max_seq_len: int, theta: float):
        """Implement the pre-norm Transformer block

            Args:
                d_model (int): Dimensionality of the Transformer block inputs
                num_heads (int): Number of heads to use in multi-head self-attention
                d_ff (int): Dimensionality of the position-wise feed-forward inner layer
                max_seq_len (int): Maximum sequence length to pre-cache if your implementation does that.
                theta (float): RoPE parameter
        """
        super().__init__()

        self.rmsnorm_first_layer = RMSNorm(d_model) # 0 matrix multiplies; 0 FLOPs | Parameter: (d_model)
        self.rmsnorm_second_layer = RMSNorm(d_model) # 0 matrix multiplies; 0 FLOPs | Parameter: (d_model)
        self.multiheadselfattentionrops_layer = MultiHeadSelfAttentionRope(d_model, num_heads, max_seq_len, theta) # 0 matrix multiplies, 0 FLOPs | Parameter: 4 * (d_model * d_model)
        self.positionwiseffn_layer = PWFFN(d_model, d_ff) # 0 matrix multiplies; 0 FLOPs | Parameter: 3 * (d_ff * d_model)
    def forward(self, x: Float[Tensor, " batch sequence_length d_model"]) -> Float[Tensor, " batch sequence_length d_model"]:
        x_norm = self.rmsnorm_first_layer.forward(x) # 0 matrix multiplies; 0 FLOPs | Parameter: 0 parameters
        # embedding_attention = self.multiheadselfattentionrops_layer.forward(x_norm)
        x_seq_len = x.size(-2)
        token_positions = torch.arange(x_seq_len).unsqueeze(0).expand(*x.shape[:-2], x_seq_len)
        embedding_attention = self.multiheadselfattentionrops_layer.forward(x_norm, token_positions) # 6 matrix multiplies; FLOPs = ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * seq_len * d_model * seq_len) + (batch...) * (2 * seq_len * seq_len * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) FLOPs |Parameters: 0 parameters

        result_firstsublayer = x + embedding_attention

        result_firstsublayer_norm = self.rmsnorm_second_layer.forward(result_firstsublayer) # 0 matrix multiplies; 0 FLOPs | Parameter: 0 parameters
        embedding_pwffn = self.positionwiseffn_layer.forward(result_firstsublayer_norm) # 3 matrix multiplies; (batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_model * d_ff) + (batch...) * (2 * 1 * d_ff * d_model) FLOPs | Parameter: 0 parameters
        result_secondsublayer = result_firstsublayer + embedding_pwffn

        return result_secondsublayer

### Norm
rmsnorm_einx.py

**element-wise multiplies:**

**dot product:**

`0 matrix multiplies`

`0 FLOPs`

**trainable parameters:** `d_model`
* init:
  * self.g(d_model)
* forward: 0 parameters

In [ ]:
import torch
import torch.nn as nn
import einx

class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-5, device: torch.device | None = None, dtype: torch.dtype | None = None):
        """ Construct the RMSNorm module.

        Args:
            d_model(int): Hidden dimension of the model
            eps(float): Epsilon value for numerical stability
            device(torch.device | None = None): Device to store the parameters on
            dtype(torch.dtype | None = None): Data type of the parameters
        """
        super().__init__()
        self.g = nn.Parameter(torch.ones(d_model, dtype=dtype, device=device)) # Parameter: self.g(d_model)
        self.eps = eps
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Process an input tensor of shape

        Args:
            x (torch.Tensor): shape (batch_size, sequence_length, d_model)
        Returns:
            torch.Tensor: shape (batch_size, sequence_length, d_model)
        """
        in_dtype = x.dtype
        x = x.to(torch.float32)

        mean_square = einx.mean('... d -> ... 1', x * x)
        inv_rms = torch.rsqrt(mean_square + self.eps)

        x_norm = einx.multiply('... d, ... 1 -> ... d', x, inv_rms)
        result = einx.multiply('... d, d -> ... d', x_norm, self.g)

        return result.to(in_dtype)


### Causal Multi-Head Self-Attention w/ RoPE
multihead_self_attention_rope.py

**element-wise multiplies:**

**dot product:**

`6 matrix multiplies`;

`((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * seq_len * d_model * seq_len) + (batch...) * (2 * seq_len * seq_len * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) FLOPs`
  * init: `0 matrix multiplies`, `0 FLOPs`
  * forward:
      * `6 matrix multiplies`
      * `FLOPs = ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * seq_len * d_model * seq_len) + (batch...) * (2 * seq_len * seq_len * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) FLOPs`

**trainable parameters:** `4 * (d_model * d_model)`
* init:
  * self.Q(d_model, d_model)
  * self.K(d_model, d_model)
  * self.V(d_model, d_model)
  * self.O(d_model, d_model)
* forward: 0 parameters

In [ ]:
import torch
import torch.nn as nn
from jaxtyping import Float, Int
from torch import Tensor
from cs336_basics.scaled_dot_product_attention import SDPAttention
from cs336_basics.rope_einx import RoPe

class MultiHeadSelfAttentionRope(nn.Module):
    def __init__(self, d_model: int, num_heads: int, max_seq_len: int, theta: float):
        """Causal multi-head self-attention

            Args:
                d_model (int): Dimensionality of the Transformer block inputs
                num_heads (int): Number of heads to use in multi-head self-attention
                max_seq_len (int): Maximum sequence length to pre-cache
                theta (float): RoPE parameter
        """
        super().__init__()

        self.Q = nn.Parameter(torch.randn(d_model, d_model)) # Parameter: self.Q(d_model, d_model)
        self.K = nn.Parameter(torch.randn(d_model, d_model)) # Parameter: self.K(d_model, d_model)
        self.V = nn.Parameter(torch.randn(d_model, d_model)) # Parameter: self.V(d_model, d_model)
        self.O = nn.Parameter(torch.randn(d_model, d_model)) # Parameter: self.O(d_model, d_model)

        self.num_heads = num_heads

        self.rope_layer = RoPe(theta=theta, d_k=d_model // num_heads, max_seq_len=max_seq_len) # 0 matrix multiplies, 0 FLOPs | Parameter: 0 parameters

    def forward(self, x: Float[Tensor, " ... sequence_length d_in"], token_positions: Int[Tensor, " ... sequence_length"] | None = None) -> Float[Tensor, " ... sequence_length d_out"]:
        batch_shape = x.shape[:-2]
        seq_len = x.shape[-2]

        q_x = x @ self.Q.T  # x(... sequence_length d_model), self.Q.T(d_model, d_model); FLOPs = ((batch...) * sequence_length) * (2 * 1 * d_model * d_model)
        k_x = x @ self.K.T  # x(... sequence_length d_model), self.K.T(d_model, d_model); FLOPs = ((batch...) * sequence_length) * (2 * 1 * d_model * d_model)
        v_x = x @ self.V.T  # x(... sequence_length d_model), self.V.T(d_model, d_model); FLOPs = ((batch...) * sequence_length) * (2 * 1 * d_model * d_model)

        q_x_heads = q_x.reshape(*batch_shape, seq_len, self.num_heads, -1)
        k_x_heads = k_x.reshape(*batch_shape, seq_len, self.num_heads, -1)
        v_x_heads = v_x.reshape(*batch_shape, seq_len, self.num_heads, -1)

        q_x_heads = q_x_heads.transpose(-3, 2)
        k_x_heads = k_x_heads.transpose(-3, 2)
        v_x_heads = v_x_heads.transpose(-3, 2)

        q_x_heads = self.rope_layer.forward(q_x_heads, token_positions) # 0 matrix multiplies, 0 FLOPs
        k_x_heads = self.rope_layer.forward(k_x_heads, token_positions) # 0 matrix multiplies, 0 FLOPs

        causal_mask = ~torch.triu(torch.ones(x.shape[-2], x.shape[-2], dtype=bool), diagonal=1)

        attention_layer = SDPAttention(q_x_heads, k_x_heads, v_x_heads, causal_mask) # 0 matrix multiplies; 0 FLOPs | Parameter: 0 parameters
        embedding_cmhsa = attention_layer.forward() # 2 matrix multiplies; (batch...) * (2 * seq_len * d_model * seq_len) + (batch...) * (2 * seq_len * seq_len * d_model) FLOPs | Parameter: 0 parameters
        embedding_cmhsa_trans = embedding_cmhsa.transpose(-3, -2)
        embedding_cmhsa_combined = embedding_cmhsa_trans.contiguous().reshape(*batch_shape, seq_len, -1)
        result = embedding_cmhsa_combined @ self.O.T # embedding_cmhsa_combined(batch..., seq_len, d_model), self.O.T(d_model, d_model); FLOPs = ((batch...) * sequence_length) * (2 * 1 * d_model * d_model)

        return result

#### Scaled Dot-Product Attention
scaled_dot_product_attention.py

**element-wise multiplies:**

**dot product:**
  * init: `0 matrix multiplies`, `0 FLOPs`
  * forward:
      * `2 matrix multiplies`
      * `(batch...) * (2 * seq_len * d_model * seq_len) + (batch...) * (2 * seq_len * seq_len * d_model) FLOPs`

**trainable parameters:** `0 parameters`

In [ ]:
import torch
import torch.nn as nn
from jaxtyping import Float
from torch import Tensor
import cs336_basics.softmax_einx as sm
import math

class SDPAttention(nn.Module):
    def __init__(self, q: Float[Tensor, "... seq_len d_k"], k: Float[Tensor, "... seq_len d_k"], v: Float[Tensor, "... seq_len d_v"], mask: Float[Tensor, " ... queries keys"] | None = None):
        super().__init__()

        self.Q = q
        self.K = k
        self.V = v
        self.mask = mask
    def forward(self) -> Float[Tensor, "... d_v"]:
        qk = self.Q @ self.K.transpose(-2, -1) # self.Q(... seq_len d_model), self.K.transpose(-2, -1)(... d_model seq_len); FLOPs = (batch...) * (2 * seq_len * d_model * seq_len)
        qk_norm = qk / math.sqrt(self.Q.size(-1))

        mask_ninf = torch.where(self.mask, torch.zeros_like(self.mask), float('-inf'))

        qk_norm_mask = qk_norm + mask_ninf

        attention_score = sm.Softmax(qk_norm_mask, -1) # 0 matrix multiplies, 0 FLOPs | Parameter: 0 parameters
        result = attention_score @ self.V # attention_score(... seq_len seq_len), self.V(... seq_len d_model); FLOPs = (batch...) * (2 * seq_len * seq_len * d_model)

        return result



##### Softmax
softmax_einx.py

**element-wise multiplies:**

**dot product:**

`0 matrix multiplies`,

`0 FLOPs`

**trainable parameters:** `0 parameters`

In [ ]:
import torch
from jaxtyping import Float
from torch import Tensor
import einx

def Softmax(x: Float[Tensor, " ..."], dim: int) -> Float[Tensor, " ..."]:
    logit_stable = einx.subtract("... logits, ... 1 -> ... logits", x, x.max(dim=dim, keepdim=True).values)
    logit_stable_exp = torch.exp(logit_stable)
    result = einx.divide("... logits, ... 1 -> ... logits", logit_stable_exp, logit_stable_exp.sum(dim=dim, keepdim=True))

    return result


#### RoPE
rope_einx.py

**element-wise multiplies:**
**dot product:**

`Num matrix multiplies: 0 matrix multiplies`

`FLOPs: 0 FLOPs`

  * init: `0 matrix multiplies` `0 FLOPs`
  * forward: `0 matrix multiplies` `0 FLOPs`

**trainable parameters:**
`0 parameters`

In [ ]:
import torch
import torch.nn as nn
from jaxtyping import Float, Int
from torch import Tensor
import einx

class RoPe(nn.Module):
    def __init__(self, theta: float, d_k: int, max_seq_len: int, device: torch.device | None = None):
        """Constructthe RoPE module and create buffers if needed.

        Args:
            theta (float): Θ value for the RoPE
            d_k (int): dimension of query and key vectors
            max_seq_len (int): Maximum sequence length that will be inputted
            device (torch.device | None): Device to store the buffer on
        """
        super().__init__()

        block_num = d_k // 2

        angle_i = torch.arange(max_seq_len)
        angle_k = torch.arange(1, block_num + 1)
        angle = einx.multiply("max_seq_len 1, 1 block_num -> max_seq_len block_num", angle_i[:, None], torch.reciprocal(theta ** ((2*angle_k[None, :] - 2) / d_k)))

        sin = torch.sin(angle)
        cos = torch.cos(angle)

        self.register_buffer("sin", sin, persistent=False)
        self.register_buffer("cos", cos, persistent=False)

    def forward(self, x: Float[Tensor, "... seq_len d_k"], token_positions: Int[Tensor, "... seq_len"]) -> Float[Tensor, "... seq_len d_k"]:
        *batch, seq_len, d_k = x.shape
        block = d_k // 2

        x_blocked = x.reshape(*batch, seq_len, block, -1)

        x_even = x_blocked[..., 0]
        x_odd = x_blocked[..., 1]

        sin_pos = self.sin[token_positions]
        cos_pos = self.cos[token_positions]

        x_even_rot = x_even * cos_pos - x_odd * sin_pos
        x_odd_rot = x_even * sin_pos + x_odd * cos_pos

        result_blocked = torch.stack((x_even_rot, x_odd_rot), dim=-1)
        result = result_blocked.reshape(*batch, seq_len, -1)

        return result


### Norm
rmsnorm_einx.py
**element-wise multiplies:**

**dot product:**

`0 matrix multiplies`

`0 FLOPs`

**trainable parameters:** `d_model`
* self.g(d_model)

In [ ]:
import torch
import torch.nn as nn
import einx

class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-5, device: torch.device | None = None, dtype: torch.dtype | None = None):
        """ Construct the RMSNorm module.

        Args:
            d_model(int): Hidden dimension of the model
            eps(float): Epsilon value for numerical stability
            device(torch.device | None = None): Device to store the parameters on
            dtype(torch.dtype | None = None): Data type of the parameters
        """
        super().__init__()
        self.g = nn.Parameter(torch.ones(d_model, dtype=dtype, device=device))
        self.eps = eps
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Process an input tensor of shape

        Args:
            x (torch.Tensor): shape (batch_size, sequence_length, d_model)
        Returns:
            torch.Tensor: shape (batch_size, sequence_length, d_model)
        """
        in_dtype = x.dtype
        x = x.to(torch.float32)

        mean_square = einx.mean('... d -> ... 1', x * x)
        inv_rms = torch.rsqrt(mean_square + self.eps)

        x_norm = einx.multiply('... d, ... 1 -> ... d', x, inv_rms)
        result = einx.multiply('... d, d -> ... d', x_norm, self.g)

        return result.to(in_dtype)


### Position-Wise Feed-Forward
positionwise_feedforward_einx.py

**element-wise multiplies:**

**dot product:**

`Num of matrix multiplies: 3 matrix multiplies`;

`FLOPs: (batch...) * (2 * seq_len * d_model * d_ff) + (batch...) * (2 * seq_len * d_model * d_ff) + (batch...) * (2 * seq_len * d_ff * d_model) FLOPs`
  * init: `0 matrix multiplies`, `0 FLOPs`
  * forward:
      * `3 matrix multiplies`
      * `(batch...) * (2 * seq_len * d_model * d_ff) + (batch...) * (2 * seq_len * d_model * d_ff) + (batch...) * (2 * seq_len * d_ff * d_model) FLOPs`

**trainable parameters:** `3 * (d_ff * d_model)`
* init:
  * self.W1(d_ff, d_model)
  * self.W3(d_ff, d_model)
  * self.W2(d_model, d_ff)
* forward: 0 parameters

In [ ]:
import torch
import torch.nn as nn
import einx
from jaxtyping import Float
from torch import Tensor

class PWFFN(nn.Module):
    def __init__(self, d_model: int, d_ff: int, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()

        # d_ff = 8/3 * d_model

        self.W1 = nn.Parameter(torch.randn(d_ff, d_model, dtype=dtype, device=device)) # Parameter: self.W1(d_ff, d_model)
        self.W3 = nn.Parameter(torch.randn(d_ff, d_model, dtype=dtype, device=device)) # Parameter: self.W3(d_ff, d_model)

        self.W2 = nn.Parameter(torch.randn(d_model, d_ff, dtype=dtype, device=device)) # Parameter: self.W2(d_model, d_ff)
    def forward(self, x: Float[Tensor, " ... d_model"]) -> Float[Tensor, "... d_model"]:
        w1_item = einx.dot("... [d_model], [d_model] d_ff -> ... d_ff", x, self.W1.T) # x(... [d_model]), self.W1.T([d_model] d_ff); FLOPs = (batch...) * (2 * 1 * d_model * d_ff)
        w1_gate_item = PWFFN.silu(w1_item) # Parameter: 0 parameter

        w3_item = einx.dot("... [d_model], [d_model] d_ff -> ... d_ff", x, self.W3.T) # x(... [d_model]), self.W3.T([d_model] d_ff); FLOPs = (batch...) * (2 * 1 * d_model * d_ff)

        l1 = einx.multiply("... d_ff, ... d_ff -> ... d_ff", w1_gate_item, w3_item)
        result = einx.dot("... [d_ff], [d_ff] d_model -> ... d_model", l1, self.W2.T) # l1(... [d_ff]), self.W2.T([d_ff] d_model); FLOPs = (batch...) * (2 * 1 * d_ff * d_model)

        return result

    @staticmethod
    def silu(x: Float[Tensor, "... d"]) -> Float[Tensor, "... d"]:
        return x * torch.sigmoid(x)

## Norm
rmsnorm_einx.py

** element-wise multiplies:**
** dot product:**

`Num of matrix multiplies: 0 matrix multiplies`

`FLOPs: 0 FLOPs`

**trainable parameters:** `d_model`
* self.g(d_model)


In [ ]:
import torch
import torch.nn as nn
import einx

class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-5, device: torch.device | None = None, dtype: torch.dtype | None = None):
        """ Construct the RMSNorm module.

        Args:
            d_model(int): Hidden dimension of the model
            eps(float): Epsilon value for numerical stability
            device(torch.device | None = None): Device to store the parameters on
            dtype(torch.dtype | None = None): Data type of the parameters
        """
        super().__init__()
        self.g = nn.Parameter(torch.ones(d_model, dtype=dtype, device=device))
        self.eps = eps
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Process an input tensor of shape

        Args:
            x (torch.Tensor): shape (batch_size, sequence_length, d_model)
        Returns:
            torch.Tensor: shape (batch_size, sequence_length, d_model)
        """
        in_dtype = x.dtype
        x = x.to(torch.float32)

        mean_square = einx.mean('... d -> ... 1', x * x)
        inv_rms = torch.rsqrt(mean_square + self.eps)

        x_norm = einx.multiply('... d, ... 1 -> ... d', x, inv_rms)
        result = einx.multiply('... d, d -> ... d', x_norm, self.g)

        return result.to(in_dtype)


## Linear(Output Embedding)
linear_module.py

**element-wise multiplies:**
**dot product:**

`Num of matrix multiplies: 1 1 matrix multiplies`

`FLOPs: (batch...) * (2 * seq_len * d_model * vocab_size) FLOPs`
  * init: `0 matrix multiplies` `0 FLOPs`
  * forward:
      * `1 matrix multiplies`
      * `FLOPs = (batch...) * (2 * seq_len * d_model * vocab_size) FLOPs`

**trainable parameters:** `vocab_size * d_model`
* init:
  * self.W(vocab_size, d_model)
* forward: 0 parameters

In [ ]:
import torch
import torch.nn as nn
import math

class Linear(nn.Module):
    def __init__(self, in_features: int, out_features: int, device: torch.device | None = None, dtype: torch.dtype | None = None):
        """Construct a linear transformation module.

        Args:
            in_features(int): final dimension of the input
            out_features(int): final dimension of the output
            device(torch.device | None): Device to store the parameters on
            dtype(torch.dtype | None): Data type of the parameters
        """
        super().__init__()
        self.W = nn.Parameter(torch.randn(out_features, in_features, dtype=dtype, device=device)) # Parameter: self.W(vocab_size, d_model)
        std_variance = math.sqrt(2/(in_features + out_features))
        nn.init.trunc_normal_(self.W, mean=0, std=std_variance, a=-3*std_variance, b=3*std_variance)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the linear transformation to the input.
        """
        return x @ self.W.T # x(batch_size, seq_len, d_model), self.W.T(d_model, vocab_size); FLOPs = (batch...) * (2 * seq_len * d_model * vocab_size)

# Q & A

## (a)
**GPT-2 XL:**
* vocab_size : 50,257
* context_length : 1,024
* num_layers : 48
* d_model : 1,600
* num_heads : 25
* d_ff : 6,400

Suppose we constructed our model using this configuration. How many trainable parameters
would our model have? Assuming each parameter is represented using single-precision floating
point, how much memory is required to just load this model?

Method: sum each component's trainable-parameter count symbolically
(embedding + per-layer Q/K/V/O + per-layer FFN + per-layer norms + final
norm + final linear, from the per-module breakdown above), simplify the
resulting expression with `sympy`, then substitute GPT-2 XL's actual
hyperparameters. For memory, multiply the resulting parameter count by 4
bytes (single-precision float32).

In [1]:
from sympy import symbols, simplify

# 1. Declare symbolic variables
vocab_size, d_model, num_layers, d_ff = symbols(
    'vocab_size d_model num_layers d_ff'
)

# 2. Define the expression
# vocab_size * d_model + num_layers * (d_model + d_model + 4 * (d_model * d_model) + 3 * (d_ff * d_model)) + d_model + vocab_size * d_model
expr = (
    vocab_size * d_model
    + num_layers * (d_model + d_model + 4 * (d_model * d_model) + 3 * (d_ff * d_model))
    + d_model
    + vocab_size * d_model
)

# 3. Simplify the symbolic expression
expr_simplified = simplify(expr)
print("Simplified symbolic expression:")
print(expr_simplified)
print()

# 4. Substitute your numeric values
values = {
    vocab_size: 50257,
    d_model: 1600,
    num_layers: 48,
    d_ff: 6400
}

parameters = expr_simplified.subs(values)

print("Num of Parameters(numeric value):")
print(parameters)

# 5. Convert to a normal Python int if needed
print("Num of Parameters(as python integer):")
print(int(parameters))

# 6. each parameter is represented using single-precision floating point, how much memory is required
# single-precision floating point: float32 = 4 bytes
mem_usage = parameters * 4

print("Memory usage(bytes):")
print(mem_usage)

Simplified symbolic expression:
d_model*(num_layers*(3*d_ff + 4*d_model + 2) + 2*vocab_size + 1)

Num of Parameters(numeric value):
2127057600
Num of Parameters(as python integer):
2127057600
Memory usage(bytes):
8508230400


## (b)
Identify the matrix multiplies required to complete a forward pass of our GPT-2 XL-shaped
model. How many FLOPs do these matrix multiplies require in total? Assume that our input
sequence has context_length tokens.
Deliverable: A list of matrix multiplies (with descriptions), and the total number of FLOPs
required.

Method: count the matrix multiplies each component does in one
forward pass (from the per-module breakdown above: 0 for Embedding/
RMSNorm/Softmax/RoPE, 6 for attention, 3 for the FFN, 1 for the final
Linear), apply the `2*m*n*p` FLOPs rule to each one, build the resulting
sum as a symbolic `sympy` expression, then substitute `context_length`
and GPT-2 XL's hyperparameters.

### In total
* Num of matrix multiplies: `num_layers * 9 + 1 = 48 * 9 + 1 = 433`
* FLOPs: `(num_layers * (((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * seq_len * d_model * seq_len) + (batch...) * (2 * seq_len * seq_len * d_model) + ((batch...) * sequence_length) * (2 * 1 * d_model * d_model) + (batch...) * (2 * seq_len * d_model * d_ff) + (batch...) * (2 * seq_len * d_model * d_ff) + (batch...) * (2 * seq_len * d_ff * d_model)))) + (batch...) * (2 * seq_len * d_model * vocab_size) FLOPs`

---

**Assume that our input sequence has context_length tokens:**
* FLOPs:
```
num_layers * (
      (context_length) * (2 * d_model * d_model)
    + (context_length) * (2 * d_model * d_model)
    + (context_length) * (2 * d_model * d_model)
    + (2 * context_length * d_model * context_length)
    + (2 * context_length * context_length * d_model)
    + (context_length) * (2 * d_model * d_model)
    + (context_length) * (2 * d_model * d_ff)
    + (context_length) * (2 * d_model * d_ff)
    + (context_length) * (2 * d_ff * d_model)
)
+ (2 * context_length * d_model * vocab_size)
```


### A list of matrix multiplies
* self.embedding_layer:
  * Num of matrix multiplies: `0`
  * FLOPs: `0`
* self.transformerblock_layer:
  * Num of matrix multiplies: `num_layers * 9`
  * FLOPs:
```
num_layers * (
  ((batch...) * sequence_length)
  * (2 * 1 * d_model * d_model)
  + ((batch...) * sequence_length)
  * (2 * 1 * d_model * d_model)
  + ((batch...) * sequence_length)
  * (2 * 1 * d_model * d_model)
  + (batch...) * (2 * seq_len * d_model * seq_len)
  + (batch...) * (2 * seq_len * seq_len * d_model)
  + ((batch...) * sequence_length)
  * (2 * 1 * d_model * d_model)
  + (batch...) * (2 * seq_len * d_model * d_ff)
  + (batch...) * (2 * seq_len * d_model * d_ff)
  + (batch...) * (2 * seq_len * d_ff * d_model)
)
```
* self.rmsnorm_layer:
  * Num of matrix multiplies: `0`
  * FLOPs: `0`
* self.linear_layer:
  * Num of matrix multiplies: `1`
  * FLOPs: `(batch...) * (2 * seq_len * d_model * vocab_size)`

---

**Assume that our input sequence has context_length tokens:**

* self.embedding_layer:
  * Num of matrix multiplies: `0`
  * FLOPs: `0`
* self.transformerblock_layer:
  * Num of matrix multiplies: `num_layers * 9`
  * FLOPs:
```
num_layers * (
  (context_length) * (2 * 1 * d_model * d_model)
  + (context_length) * (2 * 1 * d_model * d_model)
  + (context_length) * (2 * 1 * d_model * d_model)
  + (2 * context_length * d_model * context_length)
  + (2 * context_length * context_length * d_model)
  + (context_length) * (2 * 1 * d_model * d_model)
  + (context_length) * (2 * 1 * d_model * d_ff)
  + (context_length) * (2 * 1 * d_model * d_ff)
  + (context_length) * (2 * 1 * d_ff * d_model)
)
```
* self.rmsnorm_layer:
  * Num of matrix multiplies: `0`
  * FLOPs: `0`
* self.linear_layer:
  * Num of matrix multiplies: `1`
  * FLOPs: `2 * context_length * d_model * vocab_size`



In [ ]:
from sympy import symbols, simplify

# Define symbols
num_layers, context_length = symbols('num_layers context_length')
d_model, d_ff, vocab_size = symbols('d_model d_ff vocab_size')

# Build the expression
# the context_length multiplier.
FLOPs_exp = num_layers * (
      context_length * (2 * d_model * d_model)
    + context_length * (2 * d_model * d_model)
    + context_length * (2 * d_model * d_model)
    + (2 * context_length * d_model * context_length)
    + (2 * context_length * context_length * d_model)
    + context_length * (2 * d_model * d_model)
    + context_length * (2 * d_model * d_ff)
    + context_length * (2 * d_model * d_ff)
    + context_length * (2 * d_ff * d_model)
) + (2 * context_length * d_model * vocab_size)

# Simplify
FLOPs_exp_simplified = simplify(FLOPs_exp)

print("Simplified FLOPs expression:")
print(FLOPs_exp_simplified)

# Substitute your numeric values
values = {
    num_layers: 48,
    context_length: 1024,
    d_model: 1600,
    d_ff: 6400,
    vocab_size: 50257
}

FLOPs = FLOPs_exp_simplified.subs(values)

print("FLOPs:")
print(FLOPs)


Simplified FLOPs expression:
2*context_length*d_model*(num_layers*(2*context_length + 3*d_ff + 4*d_model) + vocab_size)
FLOPs:
4513336524800


## (c)
Based on your analysis above, which parts of the model require the most FLOPs?

Method: reuse the same per-component FLOPs functions as (b), but
now evaluate each one individually (not just the grand total) for one
specific config (GPT-2 XL), so each component's absolute FLOPs and its
share of the total can be compared directly.

In [ ]:
from dataclasses import dataclass
from sympy import symbols, simplify

@dataclass
class ModelConfig:
    name: str
    vocab_size: int
    context_length: int
    num_layers: int
    d_model: int
    d_ff: int

gpt2_cfg = ModelConfig(
    name="GPT-2 XL",
    vocab_size=50257,
    context_length=1024,
    num_layers=48,
    d_model=1600,
    d_ff=6400
)

def flop_embedding_layer(cfg: ModelConfig) -> int:
    return 0

def flop_transformerblock_layer(cfg: ModelConfig) -> int:
    context_length, num_layers, d_model, d_ff = symbols('context_length, num_layers, d_model, d_ff')

    exp = num_layers * (
        (context_length) * (2 * 1 * d_model * d_model)
        + (context_length) * (2 * 1 * d_model * d_model)
        + (context_length) * (2 * 1 * d_model * d_model)
        + (2 * context_length * d_model * context_length)
        + (2 * context_length * context_length * d_model)
        + (context_length) * (2 * 1 * d_model * d_model)
        + (context_length) * (2 * 1 * d_model * d_ff)
        + (context_length) * (2 * 1 * d_model * d_ff)
        + (context_length) * (2 * 1 * d_ff * d_model)
    )

    exp_simplified = simplify(exp)

    values = {
        context_length: cfg.context_length,
        num_layers: cfg.num_layers,
        d_model: cfg.d_model,
        d_ff: cfg.d_ff
    }

    flops = exp_simplified.subs(values)

    return flops

def flop_rmsnorm_layer(cfg: ModelConfig) -> int:
    return 0

def flop_linear_layer(cfg: ModelConfig) -> int:
    context_length, vocab_size, d_model = symbols('context_length, vocab_size, d_model')

    exp = 2 * context_length * d_model * vocab_size
    exp_simplified = simplify(exp)

    values = {
        context_length: cfg.context_length,
        vocab_size: cfg.vocab_size,
        d_model: cfg.d_model
    }

    flops = exp_simplified.subs(values)

    return flops

def flop_transformer_breakdown(cfg: ModelConfig) -> int:
    flops_embedding = flop_embedding_layer(cfg)
    flops_transformerblock = flop_transformerblock_layer(cfg)
    flops_norm = flop_rmsnorm_layer(cfg)
    flops_linear = flop_linear_layer(cfg)

    flops_total =  flops_embedding + flops_transformerblock + flops_norm + flops_linear

    return {
        "model": cfg.name,
        "embedding": flops_embedding,
        "transformer_block": flops_transformerblock,
        "norm": flops_norm,
        "linear": flops_linear,
        "total": flops_total
    }

import pandas as pd

row = flop_transformer_breakdown(gpt2_cfg)
df = pd.DataFrame([row]).set_index("model")

print(df)

          embedding transformer_block  norm        linear          total
model                                                                   
GPT-2 XL          0     4348654387200     0  164682137600  4513336524800


## (d)
Repeat your analysis with GPT-2 small (12 layers, 768 d_model, 12 heads), GPT-2 medium (24 layers, 1024 d_model, 16 heads), and GPT-2 large (36 layers, 1280 d_model, 20 heads). As the model size increases, which parts of the Transformer LM take up proportionally more or less of the total FLOPs?

For each model, provide a breakdown of model components and its associated FLOPs (as a proportion of the total FLOPs required for a forward pass). In addition, provide a one-to-two sentence description of how varying the model size changes the proportional FLOPs of each component.

Method: the exact same per-component breakdown function as (c),
just called once per model config (small/medium/large/XL) instead of
once. Comparing the resulting proportions across configs shows how the
split shifts as the model scales up.

In [ ]:
from dataclasses import dataclass
from sympy import symbols, simplify

@dataclass
class ModelConfig:
    name: str
    vocab_size: int
    context_length: int
    num_layers: int
    d_model: int
    d_ff: int

gpt2_xl_cfg = ModelConfig(
    name="GPT-2 XL",
    vocab_size=50257,
    context_length=1024,
    num_layers=48,
    d_model=1600,
    d_ff=6400
)

gpt2_small_cfg = ModelConfig(
    name="GPT-2 small",
    vocab_size=50257,
    context_length=1024,
    num_layers=12,
    d_model=768,
    d_ff=6400
)

gpt2_medium_cfg = ModelConfig(
    name="GPT-2 medium",
    vocab_size=50257,
    context_length=1024,
    num_layers=24,
    d_model=1024,
    d_ff=6400
)

gpt2_large_cfg = ModelConfig(
    name="GPT-2 large",
    vocab_size=50257,
    context_length=1024,
    num_layers=36,
    d_model=1280,
    d_ff=6400
)

def flop_embedding_layer(cfg: ModelConfig) -> int:
    return 0

def flop_transformerblock_layer(cfg: ModelConfig) -> int:
    context_length, num_layers, d_model, d_ff = symbols('context_length, num_layers, d_model, d_ff')

    exp = num_layers * (
        (context_length) * (2 * 1 * d_model * d_model)
        + (context_length) * (2 * 1 * d_model * d_model)
        + (context_length) * (2 * 1 * d_model * d_model)
        + (2 * context_length * d_model * context_length)
        + (2 * context_length * context_length * d_model)
        + (context_length) * (2 * 1 * d_model * d_model)
        + (context_length) * (2 * 1 * d_model * d_ff)
        + (context_length) * (2 * 1 * d_model * d_ff)
        + (context_length) * (2 * 1 * d_ff * d_model)
    )

    exp_simplified = simplify(exp)

    values = {
        context_length: cfg.context_length,
        num_layers: cfg.num_layers,
        d_model: cfg.d_model,
        d_ff: cfg.d_ff
    }

    flops = exp_simplified.subs(values)

    return flops

def flop_rmsnorm_layer(cfg: ModelConfig) -> int:
    return 0

def flop_linear_layer(cfg: ModelConfig) -> int:
    context_length, vocab_size, d_model = symbols('context_length, vocab_size, d_model')

    exp = 2 * context_length * d_model * vocab_size
    exp_simplified = simplify(exp)

    values = {
        context_length: cfg.context_length,
        vocab_size: cfg.vocab_size,
        d_model: cfg.d_model
    }

    flops = exp_simplified.subs(values)

    return flops

def flop_transformer_breakdown(cfg: ModelConfig) -> dict[str, int]:
    flops_embedding = flop_embedding_layer(cfg)
    flops_transformerblock = flop_transformerblock_layer(cfg)
    flops_norm = flop_rmsnorm_layer(cfg)
    flops_linear = flop_linear_layer(cfg)

    flops_total =  flops_embedding + flops_transformerblock + flops_norm + flops_linear

    return {
        "model": cfg.name,
        "embedding": flops_embedding,
        "transformer_block": flops_transformerblock,
        "norm": flops_norm,
        "linear": flops_linear,
        "total": flops_total
    }

def flop_transformer_breakdown_proportion(cfg: ModelConfig) -> dict[str, float | str]:
   flops_embedding = flop_embedding_layer(cfg)
   flops_transformerblock = flop_transformerblock_layer(cfg)
   flops_norm = flop_rmsnorm_layer(cfg)
   flops_linear = flop_linear_layer(cfg)
   flops_total = flops_embedding + flops_transformerblock + flops_norm + flops_linear

   flops_total_float = float(flops_total)

   return {
       "model": cfg.name,
       "embedding": f"{(flops_embedding / flops_total_float) * 100: .2f}%",
       "transformer_block": f"{(flops_transformerblock / flops_total_float) * 100: .2f}%",
       "norm": f"{(flops_norm / flops_total_float) * 100: .2f}%",
       "linear": f"{(flops_linear / flops_total_float) * 100: .2f}%",
       "total": "100%"
   }

import pandas as pd

cfg_total_model = [gpt2_small_cfg, gpt2_medium_cfg, gpt2_large_cfg, gpt2_xl_cfg]
row_value = []
row_proportion = []

for cfg in cfg_total_model:
    row_value.append(flop_transformer_breakdown(cfg))
    row_proportion.append(flop_transformer_breakdown_proportion(cfg))

df = pd.DataFrame(row_value).set_index("model")
print(df)

df = pd.DataFrame(row_proportion).set_index("model")
print(df)

              embedding transformer_block  norm        linear          total
model                                                                       
GPT-2 small           0      459024629760     0   79047426048   538072055808
GPT-2 medium          0     1275605286912     0  105396568064  1381001854976
GPT-2 large           0     2488396677120     0  131745710080  2620142387200
GPT-2 XL              0     4348654387200     0  164682137600  4513336524800
             embedding transformer_block    norm   linear total
model                                                          
GPT-2 small      0.00%            85.31%   0.00%   14.69%  100%
GPT-2 medium     0.00%            92.37%   0.00%    7.63%  100%
GPT-2 large      0.00%            94.97%   0.00%    5.03%  100%
GPT-2 XL         0.00%            96.35%   0.00%    3.65%  100%


## (e)
Take GPT-2 XL and increase the context length to 16,384. How does the total FLOPs for one forward pass change? How do the relative contribution of FLOPs of the model components change?

Method: the exact same per-component breakdown function again,
this time holding every GPT-2 XL hyperparameter fixed except
`context_length`, to isolate the effect of sequence length specifically
from the effect of model size.

In [ ]:
from dataclasses import dataclass
from sympy import symbols, simplify

@dataclass
class ModelConfig:
    name: str
    vocab_size: int
    context_length: int
    num_layers: int
    d_model: int
    d_ff: int

gpt2_xl_s_ctx_cfg = ModelConfig(
    name="GPT-2 XL(ctx_len=1024)",
    vocab_size=50257,
    context_length=1024,
    num_layers=48,
    d_model=1600,
    d_ff=6400
)

gpt2_xl_l_ctx_cfg = ModelConfig(
    name="GPT-2 XL(ctx_len=16384)",
    vocab_size=50257,
    context_length=16384,
    num_layers=48,
    d_model=1600,
    d_ff=6400
)


def flop_embedding_layer(cfg: ModelConfig) -> int:
    return 0

def flop_transformerblock_layer(cfg: ModelConfig) -> int:
    context_length, num_layers, d_model, d_ff = symbols('context_length, num_layers, d_model, d_ff')

    exp = num_layers * (
        (context_length) * (2 * 1 * d_model * d_model)
        + (context_length) * (2 * 1 * d_model * d_model)
        + (context_length) * (2 * 1 * d_model * d_model)
        + (2 * context_length * d_model * context_length)
        + (2 * context_length * context_length * d_model)
        + (context_length) * (2 * 1 * d_model * d_model)
        + (context_length) * (2 * 1 * d_model * d_ff)
        + (context_length) * (2 * 1 * d_model * d_ff)
        + (context_length) * (2 * 1 * d_ff * d_model)
    )

    exp_simplified = simplify(exp)

    values = {
        context_length: cfg.context_length,
        num_layers: cfg.num_layers,
        d_model: cfg.d_model,
        d_ff: cfg.d_ff
    }

    flops = exp_simplified.subs(values)

    return flops

def flop_rmsnorm_layer(cfg: ModelConfig) -> int:
    return 0

def flop_linear_layer(cfg: ModelConfig) -> int:
    context_length, vocab_size, d_model = symbols('context_length, vocab_size, d_model')

    exp = 2 * context_length * d_model * vocab_size
    exp_simplified = simplify(exp)

    values = {
        context_length: cfg.context_length,
        vocab_size: cfg.vocab_size,
        d_model: cfg.d_model
    }

    flops = exp_simplified.subs(values)

    return flops

def flop_transformer_breakdown(cfg: ModelConfig) -> dict[str, int]:
    flops_embedding = flop_embedding_layer(cfg)
    flops_transformerblock = flop_transformerblock_layer(cfg)
    flops_norm = flop_rmsnorm_layer(cfg)
    flops_linear = flop_linear_layer(cfg)

    flops_total =  flops_embedding + flops_transformerblock + flops_norm + flops_linear

    return {
        "model": cfg.name,
        "embedding": flops_embedding,
        "transformer_block": flops_transformerblock,
        "norm": flops_norm,
        "linear": flops_linear,
        "total": flops_total
    }

def flop_transformer_breakdown_proportion(cfg: ModelConfig) -> dict[str, float | str]:
   flops_embedding = flop_embedding_layer(cfg)
   flops_transformerblock = flop_transformerblock_layer(cfg)
   flops_norm = flop_rmsnorm_layer(cfg)
   flops_linear = flop_linear_layer(cfg)
   flops_total = flops_embedding + flops_transformerblock + flops_norm + flops_linear

   flops_total_float = float(flops_total)

   return {
       "model": cfg.name,
       "embedding": f"{(flops_embedding / flops_total_float) * 100: .2f}%",
       "transformer_block": f"{(flops_transformerblock / flops_total_float) * 100: .2f}%",
       "norm": f"{(flops_norm / flops_total_float) * 100: .2f}%",
       "linear": f"{(flops_linear / flops_total_float) * 100: .2f}%",
       "total": "100%"
   }

import pandas as pd

cfg_total_model = [gpt2_xl_s_ctx_cfg, gpt2_xl_l_ctx_cfg]
row_value = []
row_proportion = []

for cfg in cfg_total_model:
    row_value.append(flop_transformer_breakdown(cfg))
    row_proportion.append(flop_transformer_breakdown_proportion(cfg))

df = pd.DataFrame(row_value).set_index("model")
print(df.T)

print("\n" + "-" * 65 + "\n")

df = pd.DataFrame(row_proportion).set_index("model")
print(df.T)

model             GPT-2 XL(ctx_len=1024) GPT-2 XL(ctx_len=16384)
embedding                              0                       0
transformer_block          4348654387200         146887881523200
norm                                   0                       0
linear                      164682137600           2634914201600
total                      4513336524800         149522795724800

-----------------------------------------------------------------

model             GPT-2 XL(ctx_len=1024) GPT-2 XL(ctx_len=16384)
embedding                          0.00%                   0.00%
transformer_block                 96.35%                  98.24%
norm                               0.00%                   0.00%
linear                             3.65%                   1.76%
total                               100%                    100%
